In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score, brier_score_loss, mean_absolute_error

st.set_page_config(page_title='Churn decision support', page_icon='📺', layout='wide')
st.title('Customer churn decision support')
st.caption('Demonstration based on the supplied Netflix-labelled dataset; not a Netflix-operated tool or a validated operational model.')

@st.cache_data
def load_data():
    data = pd.read_csv('netflix_customer_churn.csv')
    if data['churned'].isna().any() or not set(data['churned'].unique()).issubset({0, 1}):
        raise ValueError('churned must contain only 0 and 1, without missing values.')
    return data

@st.cache_resource
def train_models(data):
    excluded = {'customer_id', 'churned', 'avg_watch_time_per_day', 'monthly_fee'}
    features = [c for c in data.columns if c not in excluded]
    X = data[features]
    y = data['churned'].astype(int)
    nums = X.select_dtypes(include='number').columns.tolist()
    cats = X.select_dtypes(exclude='number').columns.tolist()
    prep = ColumnTransformer([
        ('num', Pipeline([('fill', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), nums),
        ('cat', Pipeline([('fill', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))]), cats)
    ])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    models = {
        'Logistic regression': Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=2000))]),
        'Linear regression (benchmark)': Pipeline([('prep', prep), ('model', LinearRegression())]),
        'Random forest': Pipeline([('prep', prep), ('model', RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=42, n_jobs=-1))])
    }
    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        raw = model.predict(X_test) if name.startswith('Linear') else model.predict_proba(X_test)[:, 1]
        score = np.clip(raw, 0, 1) if name.startswith('Linear') else raw
        results.append({'Model': name, 'ROC-AUC': roc_auc_score(y_test, score),
                        'PR-AUC': average_precision_score(y_test, score),
                        'Accuracy at 0.5': accuracy_score(y_test, score >= 0.5),
                        'Brier score': brier_score_loss(y_test, score),
                        'MAE': mean_absolute_error(y_test, score)})
    return models, pd.DataFrame(results), features, nums, cats

try:
    df = load_data()
    models, report, features, nums, cats = train_models(df)
except Exception as exc:
    st.error(f'Could not prepare app: {exc}')
    st.stop()

st.info('Use the profile controls to estimate churn risk. Churn=1 means the customer churned in the dataset. The CSV does not specify a prediction horizon; verify timing before business use.')

with st.sidebar:
    st.header('Customer profile')
    inputs = {}
    for col in features:
        if col in nums:
            inputs[col] = st.number_input(col.replace('_', ' ').title(), value=float(df[col].median()), step=1.0)
        else:
            inputs[col] = st.selectbox(col.replace('_', ' ').title(), sorted(df[col].dropna().astype(str).unique()))
    threshold = st.slider('Review threshold', 0.05, 0.95, 0.50, 0.05)
    predict = st.button('Predict churn', type='primary')

if predict:
    row = pd.DataFrame([inputs])[features]
    logistic = float(models['Logistic regression'].predict_proba(row)[0, 1])
    forest = float(models['Random forest'].predict_proba(row)[0, 1])
    linear_raw = float(models['Linear regression (benchmark)'].predict(row)[0])
    a, b, c = st.columns(3)
    a.metric('Logistic churn score', f'{logistic:.1%}')
    b.metric('Random forest churn score', f'{forest:.1%}')
    c.metric('Linear benchmark (clipped)', f'{np.clip(linear_raw, 0, 1):.1%}')
    if logistic >= threshold:
        st.warning('Flag for manager review: investigate engagement and appropriate retention options. Do not automatically send an offer.')
    else:
        st.success('Below the selected review threshold; continue routine monitoring.')
    st.caption('The linear score is not a calibrated probability. A high churn score does not prove a proposed intervention will work.')

st.subheader('Holdout evaluation')
st.dataframe(report.style.format({'ROC-AUC':'{:.3f}', 'PR-AUC':'{:.3f}', 'Accuracy at 0.5':'{:.3f}', 'Brier score':'{:.3f}', 'MAE':'{:.3f}'}), hide_index=True)
st.caption(f'{len(df):,} records; stratified 75/25 train-test split, random_state=42. Churn rate: {df.churned.mean():.1%}. Lower Brier and MAE are better; higher AUC metrics are better. Revalidate with a time-based split before operational deployment.')
st.subheader('Managerial study')
st.write('Invite consenting managers or experienced decision-makers to test example profiles. Ask whether the tool is useful, understandable and trustworthy; how they handle disagreement with experience; and what would increase trust. Record responses in a secure research-approved system, not in this public demo. No participant data are collected here.')

In [ ]:
df

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,44f3ba44-b95d-4e50-a786-bac4d06f4a43,19,Female,Basic,49.17,11,Europe,Desktop,8.99,0,Credit Card,4,4.10,Drama
4996,18779bcb-ba2b-41da-b751-e70b812061ec,67,Female,Basic,9.24,2,North America,Desktop,8.99,0,PayPal,3,3.08,Documentary
4997,3f32e8c5-615b-4a3b-a864-db2688f7834f,66,Male,Standard,16.55,49,South America,Desktop,13.99,1,Debit Card,2,0.33,Action
4998,7b0ad82d-6571-430e-90f4-906259e0e89c,59,Female,Basic,9.12,3,Europe,Laptop,8.99,0,Credit Card,4,2.28,Sci-Fi


In [ ]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 63.3 MB/s eta 0:00:00
